# Workflow · criar ou atualizar o Job `v_energy_pipeline`
### V-Energy Grid 360 — Arquitetura Medalhão

Cria (ou atualiza, se já existir) o **Databricks Workflow** que roda o pipeline inteiro de ponta a ponta: ambiente → Bronze → 8 Silvers → 10 Golds → export para o banco local. É o requisito 5.1: *"orquestrar o pipeline em um Workflow do Databricks, para que ele seja reprodutível de ponta a ponta"*.

A mesma definição está em `data-engineering/workflow/v_energy_pipeline.yaml` (o arquivo que o case pede no repositório).

## As dependências

Cada tarefa só começa quando as tabelas que ela lê estão prontas. Tarefas independentes rodam em paralelo.

| Etapa | Tarefa | Depende de |
|---|---|---|
| Ambiente | `00_organizacao_do_ambiente` | — |
| Bronze | `01_ingestao_bronze` | ambiente |
| Silver | `unidades_consumidoras` | Bronze |
| | `medidores` | UCs |
| | `leituras` | medidores (remove leituras de medidores inexistentes, L2) |
| | `faturas` | leituras (uma fatura por leitura) |
| | `geracao_distribuida`, `ocorrencias_rede`, `inspecoes` | UCs |
| | `atendimentos` | UCs e leituras |
| Gold | `dim_uc` | UCs, medidores, GD e leituras |
| | `fato_consumo_faturamento` | leituras, faturas, UCs e GD |
| | `fato_atendimento`, `fato_interrupcao`, `fato_troca_medidor` | as Silvers de cada uma |
| | `receita_decomposicao` | `fato_consumo_faturamento` |
| | `kpis_mensal` | consumo, `dim_uc`, atendimento e interrupção |
| | `dec_fec_alimentador_mensal` | interrupção e `dim_uc` |
| | `fila_inspecao` | `dim_uc` e Silver de inspeções |
| | `fato_segmento_mensal` | consumo, `dim_uc`, `kpis_mensal` e `receita_decomposicao` (para as conferências do fim) |
| Export | `04_export_banco_local` | as 10 Golds |

## Agendamento

- **Mensal, no dia 3, às 03:00 (America/Recife)** — cron `0 0 3 3 * ?`. Os dados chegam por competência mensal; no começo do mês seguinte a competência anterior já está fechada (99,5% das faturas são emitidas no próprio mês, log F4), e o dia 3 dá margem para as atrasadas. De madrugada, o portal amanhece com o mês novo.
- **Pausado**: a base do case é uma extração única. O agendamento fica configurado para produção, e a execução da entrega é manual (**Run now**).
- **Reprocessamento completo** (`overwrite`): rodar de novo dá o mesmo resultado, e pagamentos atrasados atualizam faturas antigas.
- **Falha de qualidade para o pipeline:** cada notebook interrompe com erro se uma checagem falhar; as tarefas seguintes e o export não rodam, e o portal continua com os dados do mês anterior.
- **1 nova tentativa** por tarefa (para quedas momentâneas do serverless), **timeout de 1 hora** por tarefa, **uma execução por vez** e **e-mail em caso de falha** para quem criar o Job.

## Fonte dos notebooks

`USAR_GIT = True` (padrão): o Job lê os notebooks **direto da branch `main` do GitHub**, ou seja, roda exatamente o que está no repositório. **Todos os PRs precisam estar na `main`.**

`USAR_GIT = False`: lê os notebooks desta pasta do workspace (útil para testar antes do merge).

In [0]:
USAR_GIT = True   # True: notebooks da main do GitHub | False: notebooks desta pasta do workspace

import os, json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
usuario = w.current_user.me().user_name

# pasta dos notebooks no workspace (este notebook fica em data-engineering/workflow/)
pasta_notebooks = os.path.abspath(os.path.join(os.getcwd(), "..", "notebooks"))
print("usuário:", usuario)
print("pasta dos notebooks:", pasta_notebooks)

## 1. A definição do Job

A mesma do `v_energy_pipeline.yaml`. Antes de criar, o notebook confere que **todos os 21 notebooks existem** na pasta.

In [0]:
TAREFAS = [["ambiente", "00_organizacao_do_ambiente", []], ["bronze", "01_ingestao_bronze", ["ambiente"]], ["silver_ucs", "02_silver_unidades_consumidoras", ["bronze"]], ["silver_medidores", "02_silver_medidores", ["silver_ucs"]], ["silver_leituras", "02_silver_leituras", ["silver_medidores"]], ["silver_faturas", "02_silver_faturas", ["silver_leituras"]], ["silver_gd", "02_silver_geracao_distribuida", ["silver_ucs"]], ["silver_ocorrencias", "02_silver_ocorrencias_rede", ["silver_ucs"]], ["silver_inspecoes", "02_silver_inspecoes", ["silver_ucs"]], ["silver_atendimentos", "02_silver_atendimentos", ["silver_ucs", "silver_leituras"]], ["gold_dim_uc", "03_gold_dim_uc", ["silver_ucs", "silver_medidores", "silver_gd", "silver_leituras"]], ["gold_consumo_faturamento", "03_gold_fato_consumo_faturamento", ["silver_leituras", "silver_faturas", "silver_ucs", "silver_gd"]], ["gold_atendimento", "03_gold_fato_atendimento", ["silver_atendimentos", "silver_ucs"]], ["gold_interrupcao", "03_gold_fato_interrupcao", ["silver_ocorrencias", "silver_ucs"]], ["gold_troca_medidor", "03_gold_fato_troca_medidor", ["silver_leituras", "silver_medidores", "silver_ucs"]], ["gold_receita_decomposicao", "03_gold_receita_decomposicao", ["gold_consumo_faturamento"]], ["gold_kpis_mensal", "03_gold_kpis_mensal", ["gold_consumo_faturamento", "gold_dim_uc", "gold_atendimento", "gold_interrupcao"]], ["gold_dec_fec", "03_gold_dec_fec_alimentador_mensal", ["gold_interrupcao", "gold_dim_uc"]], ["gold_fila_inspecao", "03_gold_fila_inspecao", ["gold_dim_uc", "silver_inspecoes"]], ["gold_segmento_mensal", "03_gold_fato_segmento_mensal", ["gold_consumo_faturamento", "gold_dim_uc", "gold_kpis_mensal", "gold_receita_decomposicao"]], ["export_banco_local", "04_export_banco_local", ["gold_dim_uc", "gold_consumo_faturamento", "gold_atendimento", "gold_interrupcao", "gold_troca_medidor", "gold_receita_decomposicao", "gold_kpis_mensal", "gold_dec_fec", "gold_fila_inspecao", "gold_segmento_mensal"]]]
GIT_URL = "https://github.com/marcelaraposoo/V-Energy-Grid-360"
CRON, TZ = "0 0 3 3 * ?", "America/Recife"
DESCRICAO = "Pipeline V-Energy Grid 360: Bronze -> Silver -> Gold -> export para o banco local. Agendamento mensal (dia 3, 03:00, America/Recife), pausado enquanto a base for uma extração única. Cada notebook para com erro se uma checagem de qualidade falhar, e as tarefas seguintes (incluindo o export) não rodam."

existentes = {f.rsplit(".", 1)[0] for f in os.listdir(pasta_notebooks)}
faltando = [nb for _, nb, _ in TAREFAS if nb not in existentes]
if faltando:
    raise Exception(f"Notebooks não encontrados em {pasta_notebooks}: {faltando}")
print(f"{len(TAREFAS)} notebooks encontrados")

def caminho(nb):
    return f"data-engineering/notebooks/{nb}" if USAR_GIT else f"{pasta_notebooks}/{nb}"

tasks = []
for key, nb, deps in TAREFAS:
    t = {"task_key": key,
          "notebook_task": {"notebook_path": caminho(nb), "source": "GIT" if USAR_GIT else "WORKSPACE"},
          "timeout_seconds": 3600, "max_retries": 1, "min_retry_interval_millis": 60000}
    if deps:
        t["depends_on"] = [{"task_key": d} for d in deps]
    tasks.append(t)

settings = {
    "name": "v_energy_pipeline",
    "description": DESCRICAO,
    "max_concurrent_runs": 1,
    "tags": {"projeto": "v-energy-grid-360"},
    "schedule": {"quartz_cron_expression": CRON, "timezone_id": TZ, "pause_status": "PAUSED"},
    "email_notifications": {"on_failure": [usuario]},
    "tasks": tasks,
}
if USAR_GIT:
    settings["git_source"] = {"git_url": GIT_URL, "git_provider": "gitHub", "git_branch": "main"}
print(json.dumps({k: v for k, v in settings.items() if k != "tasks"}, indent=2, ensure_ascii=False))

## 2. Criar ou atualizar o Job

Se já existir um Job com o nome `v_energy_pipeline`, ele é **atualizado** (o mesmo `job_id`, o histórico de execuções fica); senão, é criado.

In [0]:
existentes = w.api_client.do("GET", "/api/2.1/jobs/list", query={"name": settings["name"]}).get("jobs", [])
if existentes:
    job_id = existentes[0]["job_id"]
    w.api_client.do("POST", "/api/2.1/jobs/reset", body={"job_id": job_id, "new_settings": settings})
    print("Job atualizado:", job_id)
else:
    job_id = w.api_client.do("POST", "/api/2.1/jobs/create", body=settings)["job_id"]
    print("Job criado:", job_id)

print("Abra em: Jobs & Pipelines -> v_energy_pipeline  |  ", f"{w.config.host}/jobs/{job_id}")

## 3. Próximos passos (na tela do Job)

1. **Run now** e esperar todas as tarefas ficarem verdes.
2. **Print da execução bem-sucedida** (aba *Runs* → a execução → visão em grafo): salvar como `docs/img/workflow_execucao.png` e colocar no README (obrigatório pelo case).
3. Conferir o **agendamento** em *Schedules & Triggers*: mensal, dia 3, 03:00, `America/Recife`, **pausado**.
4. Se mudar algo na tela, exportar de novo o YAML (*⋮ → View as code → YAML*) para `data-engineering/workflow/v_energy_pipeline.yaml`.